<a href="https://colab.research.google.com/github/srilakshmi-saladi/unet/blob/main/DR_Novel_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EfficientNetB3 — Novel DR Grading (QWK target > 0.88)

### Novel contributions vs published work
| # | Contribution | Why it's novel |
|---|---|---|
| 1 | **Ordinal Regression Head** | Most papers use softmax; this treats grades as ordered |
| 2 | **External CBAM Attention** | Injected *after* backbone — distinct from SE inside EfficientNet |
| 3 | **MC Dropout Uncertainty** | Flag unreliable predictions for clinical review |
| 4 | **Compound-Ordinal Loss** | Focal BCE + distance penalty for large grade jumps |
| 5 | **Test-Time Augmentation** | 8-pass ensemble at inference |

**Dataset:** IDRiD (Disease Grading subset)

In [3]:
# ═══════════════════════════════════════════════════
# CELL 1 — Install & imports
# ═══════════════════════════════════════════════════
!pip -q install opencv-python scikit-learn pandas

import os, glob, zipfile
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
# ═══════════════════════════════════════════════════
# CELL 2 — Mount Drive & unzip IDRiD
# ═══════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

# ── CHANGE THIS to your zip location in Drive ──────
ZIP_PATH = "/content/drive/MyDrive/fundusDatasets/DR Grading/B_Disease Grading.zip"

OUT_DIR = "/content/idrid_grading"
os.makedirs(OUT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(OUT_DIR)
print('Unzipped to:', OUT_DIR)

ROOT = OUT_DIR + '/B. Disease Grading'
TRAIN_IMG_DIR = ROOT + '/1. Original Images/a. Training Set'
TEST_IMG_DIR  = ROOT + '/1. Original Images/b. Testing Set'

train_imgs = glob.glob(TRAIN_IMG_DIR + '/*.jpg')
test_imgs  = glob.glob(TEST_IMG_DIR  + '/*.jpg')
print(f'Train images: {len(train_imgs)} | Test images: {len(test_imgs)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Unzipped to: /content/idrid_grading
Train images: 413 | Test images: 103


In [5]:
# ═══════════════════════════════════════════════════
# CELL 3 — Auto-detect CSV & build train_df / val_df
# ═══════════════════════════════════════════════════
csvs = sorted(glob.glob(ROOT + '/2. Groundtruths/**/*.csv', recursive=True))
print('CSVs found:', csvs)

def pick_train_csv(csv_list):
    for c in csv_list:
        if 'train' in os.path.basename(c).lower():
            return c
    sizes = sorted([(c, len(pd.read_csv(c))) for c in csv_list],
                   key=lambda x: x[1], reverse=True)
    return sizes[0][0]

TRAIN_CSV = pick_train_csv(csvs)
print('Using:', TRAIN_CSV)

df_raw = pd.read_csv(TRAIN_CSV)
print('Columns:', df_raw.columns.tolist())
print(df_raw.head(3))

# Auto-detect column names
IMG_COLS   = ['Image name', 'Image', 'image', 'img', 'filename', 'File name']
GRADE_COLS = ['Retinopathy grade', 'DR grade', 'grade', 'dr_grade', 'retinopathy']

image_col = next((c for c in IMG_COLS   if c in df_raw.columns), df_raw.columns[0])
grade_col = next((c for c in GRADE_COLS if c in df_raw.columns), None)
if grade_col is None:
    grade_col = next((c for c in df_raw.columns if 'grade' in c.lower()), None)
if grade_col is None:
    raise ValueError('Cannot detect grade column. Set grade_col manually.')

print(f'image_col={image_col} | grade_col={grade_col}')

def ensure_ext(x):
    x = str(x).strip()
    return x if x.lower().endswith(('.jpg','.jpeg','.png')) else x + '.jpg'

df_raw['filename'] = df_raw[image_col].apply(ensure_ext)
df_raw['label']    = df_raw[grade_col].astype(int)
df_raw['path']     = df_raw['filename'].apply(
    lambda x: os.path.join(TRAIN_IMG_DIR, x))

df_raw = df_raw[df_raw['path'].apply(os.path.exists)].reset_index(drop=True)
print(f'Matched images: {len(df_raw)}')
print('Label dist:\n', df_raw['label'].value_counts().sort_index())

SEED = 42
train_df, val_df = train_test_split(
    df_raw, test_size=0.20, random_state=SEED, stratify=df_raw['label']
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

CSVs found: ['/content/idrid_grading/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv', '/content/idrid_grading/B. Disease Grading/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv']
Using: /content/idrid_grading/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv
Columns: ['Image name', 'Retinopathy grade', 'Risk of macular edema ', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']
  Image name  Retinopathy grade  Risk of macular edema   Unnamed: 3  \
0  IDRiD_001                  3                       2         NaN   
1  IDRiD_002                  3                       2         NaN   
2  IDRiD_003                  2                       2         NaN   

   Unnamed: 4  Unnamed: 5  Unnamed: 6  Unnamed: 7  Unnamed: 8  Unnamed: 9  \
0         NaN         NaN         NaN         NaN         NaN         NaN   
1         NaN         NaN   

In [7]:
# ONE-TIME CACHE — run once, takes ~4 min, never again
import os
import numpy as np
from tqdm.notebook import tqdm

CACHE_DIR = '/content/idrid_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# ── paste the preprocessing inline so this cell is self-contained ──
def _preprocess_for_cache(path, img_size):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f'Cannot read: {path}')
    # crop black border
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > 7
    if mask.any():
        coords = np.argwhere(mask)
        y0, x0 = coords.min(axis=0)
        y1, x1 = coords.max(axis=0) + 1
        img = img[y0:y1, x0:x1]
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # illumination correction + CLAHE on green channel
    g  = img[:, :, 1].astype(np.float32)
    bg = cv2.GaussianBlur(g, (0,0), sigmaX=30, sigmaY=30)
    g_corr = np.clip(cv2.addWeighted(g, 1.0, bg, -1.0, 128.0), 0, 255).astype(np.uint8)
    clahe  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    g_clahe = clahe.apply(g_corr)
    # skip fastNlMeansDenoising — it's the slow culprit, not needed
    img[:, :, 1] = g_clahe
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return img.astype(np.float32)

all_paths = pd.concat([train_df, val_df])['path'].unique()
missing = [p for p in all_paths
           if not os.path.exists(os.path.join(CACHE_DIR, os.path.basename(p) + '.npy'))]

if len(missing) == 0:
    print('✅ Already cached. Skipping.')
else:
    print(f'Caching {len(missing)} images...')
    errors = []
    for p in tqdm(missing):
        try:
            img = _preprocess_for_cache(p, IMG_SIZE)
            np.save(os.path.join(CACHE_DIR, os.path.basename(p) + '.npy'), img)
        except Exception as e:
            errors.append((p, str(e)))
    print(f'Done. Cached: {len(missing)-len(errors)}  Errors: {len(errors)}')

train_df['cache_path'] = train_df['path'].apply(
    lambda p: os.path.join(CACHE_DIR, os.path.basename(p) + '.npy'))
val_df['cache_path'] = val_df['path'].apply(
    lambda p: os.path.join(CACHE_DIR, os.path.basename(p) + '.npy'))
print('cache_path column added to train_df and val_df')

Caching 413 images...


  0%|          | 0/413 [00:00<?, ?it/s]

Done. Cached: 0  Errors: 413
cache_path column added to train_df and val_df


In [8]:
# ═══════════════════════════════════════════════════
# CELL 4 — Config
# ═══════════════════════════════════════════════════
NUM_CLASSES      = 5
BATCH_SIZE       = 8      # Use 16 on A100/V100; keep 8 for T4
IMG_SIZE         = 384    # Drop to 299 if OOM
EPOCHS_A         = 10
EPOCHS_B         = 30
FINE_TUNE_LAST_N = 80
STEPS_PER_EPOCH  = 60
MC_SAMPLES       = 20
AUTOTUNE         = tf.data.AUTOTUNE

tf.random.set_seed(SEED)
np.random.seed(SEED)
print('Config OK')

Config OK


In [9]:
# ═══════════════════════════════════════════════════
# CELL 5 — Preprocessing (CLAHE + illumination correction)
# ═══════════════════════════════════════════════════
def crop_black_border(img_bgr, tol=7):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    mask = gray > tol
    if mask.any():
        coords = np.argwhere(mask)
        y0, x0 = coords.min(axis=0)
        y1, x1 = coords.max(axis=0) + 1
        return img_bgr[y0:y1, x0:x1]
    return img_bgr

def illumination_correction_green(img_rgb, sigma=30):
    g  = img_rgb[:, :, 1].astype(np.float32)
    bg = cv2.GaussianBlur(g, (0,0), sigmaX=sigma, sigmaY=sigma)
    corrected = np.clip(cv2.addWeighted(g, 1.0, bg, -1.0, 128.0), 0, 255).astype(np.uint8)
    return corrected

def apply_fundus_preprocess(img_rgb):
    g_corr  = illumination_correction_green(img_rgb)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    g_clahe = clahe.apply(g_corr)
    g_clahe = cv2.fastNlMeansDenoising(g_clahe, None, h=5,
                  templateWindowSize=7, searchWindowSize=21)
    out = img_rgb.copy()
    out[:, :, 1] = g_clahe
    return out

def load_and_preprocess(path, img_size):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f'Cannot read: {path}')
    img = crop_black_border(img)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = apply_fundus_preprocess(img)
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return img.astype(np.float32)

print('Preprocessing functions ready')

Preprocessing functions ready


In [11]:
# ═══════════════════════════════════════════════════
# CELL 6 — Ordinal encoding
# Grade k  →  [1,1,..,1, 0,0,..,0]  (k ones, length K-1)
# Decoding →  count outputs > 0.5
# ═══════════════════════════════════════════════════
def grade_to_ordinal(grade, num_classes=5):
    vec = np.zeros(num_classes - 1, dtype=np.float32)
    vec[:int(grade)] = 1.0
    return vec

def ordinal_to_grade(sigmoid_out):
    return np.sum(np.array(sigmoid_out) > 0.5, axis=-1)

# Quick sanity check
for g in range(5):
    enc = grade_to_ordinal(g)
    dec = ordinal_to_grade(enc)
    assert dec == g, f'Mismatch at grade {g}'
print('Ordinal encoding OK  (grade → vector → grade)')

Ordinal encoding OK  (grade → vector → grade)


In [13]:
# ═══════════════════════════════════════════════════
# CELL 7 — tf.data pipeline (balanced oversampling + MixUp)
# ═══════════════════════════════════════════════════
augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomFlip('vertical'),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.20),
    layers.RandomBrightness(0.15),
], name='augment')

def tf_load_ordinal(path, label):
    def _fn(p, lbl):
        img = np.load(p.numpy().decode()).astype(np.float32) / 255.0
        y   = grade_to_ordinal(int(lbl.numpy()))
        return img, y
    img, y = tf.py_function(_fn, [path, label], Tout=[tf.float32, tf.float32])
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    y.set_shape([NUM_CLASSES - 1])
    return img, y
def mixup_batch(x, y, alpha=0.2):
    bs  = tf.shape(x)[0]
    a   = tf.random.gamma([bs], alpha, beta=1.0)
    b   = tf.random.gamma([bs], alpha, beta=1.0)
    lam = a / (a + b)
    lx  = tf.reshape(lam, [bs, 1, 1, 1])
    ly  = tf.reshape(lam, [bs, 1])
    idx = tf.random.shuffle(tf.range(bs))
    return (x * lx + tf.gather(x, idx) * (1 - lx),
            y * ly + tf.gather(y, idx) * (1 - ly))

def make_train_ds(df):
    counts    = df['label'].value_counts().to_dict()
    max_count = max(counts.values())
    class_ds  = []
    for c in range(NUM_CLASSES):
        sub = df[df['label'] == c]
        if len(sub) == 0: continue
        ds = tf.data.Dataset.from_tensor_slices(
            (sub['cache_path'].values, sub['label'].values)
        ).shuffle(len(sub), seed=SEED, reshuffle_each_iteration=True).repeat()
        class_ds.append(ds)
    ds = tf.data.Dataset.sample_from_datasets(
        class_ds, weights=[1.0]*len(class_ds), seed=SEED)
    ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(tf_load_ordinal, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (augment(x, training=True), y),
                num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=True)
    ds = ds.map(mixup_batch, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

def make_val_ds(df):
    ds = tf.data.Dataset.from_tensor_slices(
        (df['cache_path'].values, df['label'].values))
    ds = ds.map(tf_load_ordinal, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_train_ds(train_df)
val_ds   = make_val_ds(val_df)
print('Datasets ready.')

Datasets ready.


In [14]:
# ═══════════════════════════════════════════════════
# CELL 8 — CBAM Attention (novel: external to backbone)
# ═══════════════════════════════════════════════════
class ChannelAttention(layers.Layer):
    def __init__(self, ratio=8, **kw):
        super().__init__(**kw)
        self.ratio = ratio

    def build(self, input_shape):
        C = input_shape[-1]
        self.gap = layers.GlobalAveragePooling2D()
        self.gmp = layers.GlobalMaxPooling2D()
        self.fc1 = layers.Dense(C // self.ratio, activation='relu', use_bias=False)
        self.fc2 = layers.Dense(C, use_bias=False)

    def call(self, x):
        avg   = self.fc2(self.fc1(self.gap(x)))
        mx    = self.fc2(self.fc1(self.gmp(x)))
        scale = tf.reshape(tf.sigmoid(avg + mx), [-1, 1, 1, tf.shape(avg)[-1]])
        return x * scale

    def get_config(self):
        cfg = super().get_config(); cfg['ratio'] = self.ratio; return cfg


class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kw):
        super().__init__(**kw)
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.conv = layers.Conv2D(1, self.kernel_size, padding='same',
                                  activation='sigmoid', use_bias=False)

    def call(self, x):
        avg   = tf.reduce_mean(x, axis=-1, keepdims=True)
        mx    = tf.reduce_max(x,  axis=-1, keepdims=True)
        return x * self.conv(tf.concat([avg, mx], axis=-1))

    def get_config(self):
        cfg = super().get_config(); cfg['kernel_size'] = self.kernel_size; return cfg


class CBAM(layers.Layer):
    """Channel then Spatial attention — applied EXTERNALLY after EfficientNet."""
    def __init__(self, ratio=8, kernel_size=7, **kw):
        super().__init__(**kw)
        self.ratio = ratio; self.kernel_size = kernel_size
        self.ca = ChannelAttention(ratio=ratio)
        self.sa = SpatialAttention(kernel_size=kernel_size)

    def call(self, x):
        return self.sa(self.ca(x))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'ratio': self.ratio, 'kernel_size': self.kernel_size})
        return cfg

CUSTOM_OBJECTS = {'CBAM': CBAM, 'ChannelAttention': ChannelAttention,
                  'SpatialAttention': SpatialAttention}
print('CBAM layers defined.')

CBAM layers defined.


In [15]:
# ═══════════════════════════════════════════════════
# CELL 9 — Compound-Ordinal Loss
# = Focal BCE on each ordinal threshold
# + squared distance penalty for large grade jumps
# ═══════════════════════════════════════════════════
def compound_ordinal_loss(y_true, y_pred_logits):
    # Part 1: binary focal cross-entropy on K-1 thresholds
    bce = tf.keras.losses.binary_focal_crossentropy(
        y_true, y_pred_logits,
        apply_class_balancing=False,
        gamma=2.0,
        from_logits=True
    )
    # Part 2: penalise large ordinal jumps
    sig        = tf.sigmoid(y_pred_logits)
    true_grade = tf.cast(tf.reduce_sum(y_true, axis=-1), tf.float32)
    pred_grade = tf.reduce_sum(tf.cast(sig > 0.5, tf.float32), axis=-1)
    dist_pen   = tf.reduce_mean(tf.square(tf.abs(true_grade - pred_grade)) * 0.1)
    return tf.reduce_mean(bce) + dist_pen

CUSTOM_OBJECTS['compound_ordinal_loss'] = compound_ordinal_loss
print('Loss function defined.')

Loss function defined.


In [16]:
# ═══════════════════════════════════════════════════
# CELL 10 — Build model
# EfficientNetB3 → CBAM → MC Dropout → Ordinal head
# ═══════════════════════════════════════════════════
def build_model():
    inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    base    = keras.applications.EfficientNetB3(
                  include_top=False, weights='imagenet')
    preproc = keras.applications.efficientnet.preprocess_input

    x = preproc(inputs * 255.0)
    x = base(x, training=False)                           # frozen backbone
    x = CBAM(ratio=8, kernel_size=7, name='cbam')(x)     # ← novel: external attention
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.50, name='mc_dropout')(x)        # ← MC Dropout node
    x = layers.Dense(256, activation='swish')(x)
    x = layers.Dropout(0.30)(x)
    # Ordinal head: K-1 raw logits, NO softmax/sigmoid here
    out = layers.Dense(NUM_CLASSES - 1, activation=None,
                        name='ordinal_logits')(x)
    return keras.Model(inputs, out), base

model, base = build_model()
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply (Multiply)             │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 12, 12, 1536)   │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cbam (CBAM)                     │ (None, 12, 12, 1536)   │       589,922 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1536)           │         6,144 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mc_dropout (Dropout)            │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       393,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ordinal_logits (Dense)          │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,774,101 (44.91 MB)

 Trainable params: 11,683,726 (44.57 MB)

 Non-trainable params: 90,375 (353.03 KB)

In [17]:
# ═══════════════════════════════════════════════════
# CELL 11 — QWK Callback (ordinal-aware)
# ═══════════════════════════════════════════════════
class OrdinalQWKCallback(keras.callbacks.Callback):
    def __init__(self, val_ds, save_path='best_qwk.keras'):
        super().__init__()
        self.val_ds   = val_ds
        self.save_path = save_path
        self.best_qwk  = -1.0

    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []
        for xb, yb in self.val_ds:
            logits = self.model.predict(xb, verbose=0)
            y_pred.extend(ordinal_to_grade(tf.sigmoid(logits).numpy()))
            y_true.extend(ordinal_to_grade(yb.numpy()))
        qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
        print(f'  val_QWK: {qwk:.4f}')
        if qwk > self.best_qwk:
            self.best_qwk = qwk
            self.model.save(self.save_path)
            print(f'  ✅ Saved best → {self.save_path}  (QWK={qwk:.4f})')

print('Callback defined.')

Callback defined.


In [18]:
# ═══════════════════════════════════════════════════
# CELL 12 — Stage A: Train head only (backbone frozen)
# ═══════════════════════════════════════════════════
base.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=compound_ordinal_loss
)

cbs_a = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    OrdinalQWKCallback(val_ds, save_path='best_stageA.keras'),
]

print('\n=== Stage A: Head training ===')
hist_a = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_A,
    steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=cbs_a,
)


=== Stage A: Head training ===
Epoch 1/10


UnknownError: Graph execution error:

Detected at node EagerPyFunc defined at (most recent call last):
<stack traces unavailable>
Detected at node EagerPyFunc defined at (most recent call last):
<stack traces unavailable>
2 root error(s) found.
  (0) UNKNOWN:  Error in user-defined function passed to ParallelMapDatasetV2:60 transformation with iterator: Iterator::Root::Prefetch::ParallelMapV2::MapAndBatch::ParallelMapV2: FileNotFoundError: [Errno 2] No such file or directory: '/content/idrid_cache/IDRiD_122.jpg.npy'
Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/ops/script_ops.py", line 267, in __call__
    return func(device, token, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^

  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/ops/script_ops.py", line 145, in __call__
    outputs = self._call(device, args)
              ^^^^^^^^^^^^^^^^^^^^^^^^

  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/ops/sc

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 13 — Stage B: Fine-tune last N layers
# ═══════════════════════════════════════════════════
base.trainable = True

# Freeze everything except last FINE_TUNE_LAST_N layers
for layer in base.layers[:-FINE_TUNE_LAST_N]:
    layer.trainable = False

# Always freeze BatchNorm statistics (critical for small batches)
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(1 for l in base.layers if l.trainable)
print(f'Trainable backbone layers: {trainable_count}')

total_steps = STEPS_PER_EPOCH * EPOCHS_B
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=5e-5,
    decay_steps=total_steps,
    alpha=1e-6
)

model.compile(
    optimizer=keras.optimizers.AdamW(lr_schedule, weight_decay=1e-4),
    loss=compound_ordinal_loss
)

cbs_b = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=8, restore_best_weights=True),
    OrdinalQWKCallback(val_ds, save_path='best_qwk.keras'),
]

print('\n=== Stage B: Fine-tuning ===')
hist_b = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_B,
    steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=cbs_b,
)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 14 — TTA Inference (8-pass ensemble)
# ═══════════════════════════════════════════════════
tta_aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.08),
], name='tta')

def tta_predict(mdl, x_batch, n_passes=8):
    sig_sum = None
    for _ in range(n_passes):
        logits  = mdl(tta_aug(x_batch, training=True), training=False)
        sig     = tf.sigmoid(logits).numpy()
        sig_sum = sig if sig_sum is None else sig_sum + sig
    return ordinal_to_grade(sig_sum / n_passes)

print('TTA function ready.')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 15 — MC Dropout Uncertainty Estimation
# Novel: flag unreliable predictions for specialist review
# ═══════════════════════════════════════════════════
def ordinal_grade_probs(sig):
    """Convert ordinal sigmoid outputs to P(grade=k) for each k."""
    K  = sig.shape[-1] + 1
    ps = [1 - sig[:, 0]]
    for k in range(1, K - 1):
        ps.append(sig[:, k-1] - sig[:, k])
    ps.append(sig[:, -1])
    return np.stack(ps, axis=-1).clip(0, 1)

def mc_dropout_predict(mdl, x_batch, n_samples=MC_SAMPLES):
    """
    Returns:
      mean_grade   — predicted grade (most probable)
      entropy      — predictive entropy (higher = less certain)
      mean_probs   — grade probabilities
    """
    all_sigs = []
    for _ in range(n_samples):
        logits = mdl(x_batch, training=True)  # training=True keeps dropout on
        all_sigs.append(tf.sigmoid(logits).numpy())
    all_sigs = np.stack(all_sigs, axis=0)     # (MC, B, K-1)

    mean_sig   = all_sigs.mean(axis=0)
    mean_grade = ordinal_to_grade(mean_sig)

    probs_per_sample = np.array([ordinal_grade_probs(all_sigs[i])
                                  for i in range(n_samples)])  # (MC, B, K)
    mean_probs = np.clip(probs_per_sample.mean(axis=0), 1e-9, 1.0)  # (B, K)

    # Predictive entropy: H = -sum(p * log p)
    entropy = -np.sum(mean_probs * np.log(mean_probs), axis=-1)     # (B,)
    return mean_grade, entropy, mean_probs

print('MC Dropout uncertainty functions ready.')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 16 — Full Evaluation (TTA + MC Dropout)
# ═══════════════════════════════════════════════════
best_model = keras.models.load_model(
    'best_qwk.keras',
    custom_objects=CUSTOM_OBJECTS
)

val_ds_eval   = make_val_ds(val_df)
y_true_all    = []
y_pred_tta    = []
y_pred_mc     = []
uncertainties = []

print('Evaluating...')
for x_batch, y_batch in val_ds_eval:
    trues = ordinal_to_grade(y_batch.numpy())
    y_true_all.extend(trues)

    y_pred_tta.extend(tta_predict(best_model, x_batch, n_passes=8))

    mc_grades, entropy, _ = mc_dropout_predict(best_model, x_batch)
    y_pred_mc.extend(mc_grades)
    uncertainties.extend(entropy)

y_true_all    = np.array(y_true_all)
y_pred_tta    = np.array(y_pred_tta)
y_pred_mc     = np.array(y_pred_mc)
uncertainties = np.array(uncertainties)

qwk_tta = cohen_kappa_score(y_true_all, y_pred_tta, weights='quadratic')
qwk_mc  = cohen_kappa_score(y_true_all, y_pred_mc,  weights='quadratic')
acc_tta = (y_true_all == y_pred_tta).mean()
severe  = np.mean(np.abs(y_true_all - y_pred_tta) >= 2)

# Confident-only QWK (top 75% certain predictions)
thresh     = np.percentile(uncertainties, 75)
conf_mask  = uncertainties < thresh
qwk_conf   = cohen_kappa_score(
    y_true_all[conf_mask], y_pred_mc[conf_mask], weights='quadratic'
) if conf_mask.sum() > 5 else 0.0

print('\n══════════════ FINAL RESULTS ══════════════')
print(f'  QWK  (TTA 8-pass):              {qwk_tta:.4f}')
print(f'  QWK  (MC Dropout mean):         {qwk_mc:.4f}')
print(f'  QWK  (MC, top-75% confident):   {qwk_conf:.4f}')
print(f'  Accuracy (TTA):                 {acc_tta:.4f}')
print(f'  Severe error rate |diff|≥2:     {severe:.4f}')
print(f'  Mean uncertainty (entropy):     {uncertainties.mean():.4f}')
print(f'  Cases flagged for review:       {(~conf_mask).sum()} / {len(y_true_all)}')
print('═══════════════════════════════════════════')

print('\nPer-class breakdown:')
for c in range(NUM_CLASSES):
    m = y_true_all == c
    if m.sum() > 0:
        print(f'  Grade {c}: acc={( y_pred_tta[m]==c).mean():.3f}'
              f'  mean_uncertainty={uncertainties[m].mean():.3f}'
              f'  n={m.sum()}')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 17 — Save to Drive
# ═══════════════════════════════════════════════════
SAVE_DIR = '/content/drive/MyDrive/DR_Novel_Results'
os.makedirs(SAVE_DIR, exist_ok=True)

best_model.save(f'{SAVE_DIR}/best_qwk_final.keras')

results = pd.DataFrame({
    'y_true':        y_true_all,
    'y_pred_tta':    y_pred_tta,
    'y_pred_mc':     y_pred_mc,
    'uncertainty':   uncertainties,
    'flagged_review': (~conf_mask).astype(int)
})
results.to_csv(f'{SAVE_DIR}/predictions.csv', index=False)
print(f'Saved model + predictions to {SAVE_DIR}')